# Query RAG System

Sends each QA pair to the RAG system (vector retrieval + LLM generation) and saves the results as `eval_data_health_wallet.json`.
This file is consumed by the metric evaluation notebooks — run once per experiment.

**Prerequisites:**
- `../qa_pairs.json` generated by `prepare_qa_pairs.ipynb`

**Output:**
- `eval_data_health_wallet.json` — list of `{user_input, response, retrieved_contexts, reference}`

## Setup

In [1]:
!pip install -q rich

In [2]:
import json
import os
import re

import requests

In [3]:
# --- Configuration ---
RAG_URL = "http://llamastack.llama-stack-rag.svc.cluster.local:8321"
VECTOR_STORE_NAME = "vszp-health-wallet-vector-store-v2"
TOP_K = 4

In [4]:
print("RAG system models:")
rag_models_resp = requests.get(f"{RAG_URL}/v1/models").json()
rag_models = rag_models_resp["data"]
for m in rag_models:
    print(f"  {m['identifier']} ({m['model_type']})")

RAG system models:
  qwen-3-4b-embedding/qwen3-4b-embedding (embedding)
  gemma-4-31b-it/Gemma-4-31B-IT (llm)
  sentence-transformers/nomic-ai/nomic-embed-text-v1.5 (embedding)


## Load QA Pairs

In [5]:
with open("../qa_pairs.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

qa_pairs = qa_data["qa_pairs"]
pdf_key = qa_data["source_pdf"]
print(f"Loaded {len(qa_pairs)} Q&A pairs from {pdf_key}")
for i, qa in enumerate(qa_pairs[:3]):
    print(f"\nQ{i+1}: {qa['question']}")
    print(f"A{i+1}: {qa['reference'][:120]}...")

Loaded 182 Q&A pairs from evaldata_najcastejsie-otazky-penazenka-zdravia.pdf

Q1: Kde nájdem Peňaženku zdravia? Je spoplatnená?
A1: Peňaženku zdravia nájdete vo voľne dostupnej mobilnej aplikácii VšZP a v ePobočke. Využívať ju môžete po zaregistrovaní ...

Q2: Prečo je Peňaženka zdravia viazaná na mobilnú aplikáciu?
A2: Mobilná aplikácia vám umožní rýchly prístup k Peňaženke zdravia, kdekoľvek sa nachádzate. Pokiaľ vám viac vyhovuje webov...

Q3: Na čerpanie príspevkov z Peňaženky zdravia musím mať mobilnú aplikáciu,
alebo mi stačí ePobočka?
A3: Na čerpanie príspevkov z Peňaženky zdravia stačí, ak máte zriadenú ePobočku....


## Query RAG System

For each question, retrieve relevant contexts from the vector store and generate a response using the LLM.

In [6]:
# Find the vector store UUID
vs_response = requests.get(f"{RAG_URL}/v1/vector_stores")
vector_stores = vs_response.json()["data"]
vs_id = None
for vs in vector_stores:
    if vs["name"] == VECTOR_STORE_NAME:
        vs_id = vs["id"]
        break

assert vs_id, f"Vector store '{VECTOR_STORE_NAME}' not found!"
print(f"Vector store: {VECTOR_STORE_NAME}")
print(f"UUID: {vs_id}")

# Find LLM model ID
llm_model_id = None
for m in rag_models:
    if m["model_type"] == "llm":
        llm_model_id = m["identifier"]
        break

assert llm_model_id, "No LLM model found in RAG system!"
print(f"LLM model: {llm_model_id}")

Vector store: vszp-health-wallet-vector-store-v2
UUID: vs_d7b17dd5-34e7-4778-a7c9-f9ae96d7a404
LLM model: gemma-4-31b-it/Gemma-4-31B-IT


In [7]:
# Query RAG system for each question (resumable + auto-save)
if os.path.exists("eval_data_health_wallet.json"):
    with open("eval_data_health_wallet.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    print(f"Resuming from {len(eval_data)} existing entries")
else:
    eval_data = []
errors = []

print(f"Querying RAG system for {len(qa_pairs)} questions (starting from {len(eval_data)})...\n")

for i, qa in enumerate(qa_pairs):
    if i < len(eval_data):
        continue

    question = qa["question"]
    reference = qa["reference"]
    print(f"[{i+1}/{len(qa_pairs)}] {question[:80]}...", end=" ")

    try:
        query_resp = requests.post(
            f"{RAG_URL}/v1/vector-io/query",
            json={"vector_db_id": vs_id, "query": question, "params": {"max_chunks": TOP_K}},
        )
        chunks = query_resp.json().get("chunks", [])
        retrieved_contexts = []
        for c in chunks:
            text = c.get("text", c.get("content", ""))
            if isinstance(text, dict):
                text = text.get("text", str(text))
            if text:
                retrieved_contexts.append(text)

        context_text = "\n\n---\n\n".join(retrieved_contexts)
        system_prompt = (
            "Si špecializovaný asistent pre benefity Peňaženky zdravia (MINI a MAXI)"
            "od Všeobecnej zdravotnej poisťovne. "
            "Odpovedaj výhradne na základe poskytnutého kontextu. "
            "Odpovedaj vždy v slovenčine, profesionálne a vecne.\n\n"
            f"Kontext:\n{context_text}"
        )

        chat_resp = requests.post(
            f"{RAG_URL}/v1/chat/completions",
            json={
                "model": llm_model_id,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": question},
                ],
                "max_tokens": 1024,
                "temperature": 0.1,
            },
        )
        response_text = chat_resp.json()["choices"][0]["message"]["content"]
        response_text = re.sub(r"<think>.*?</think>\s*", "", response_text, flags=re.DOTALL)

        eval_data.append({
            "user_input": question,
            "response": response_text,
            "retrieved_contexts": retrieved_contexts,
            "reference": reference,
        })
        print(f"OK ({len(retrieved_contexts)} contexts, {len(response_text)} chars)")

        with open("eval_data_health_wallet.json", "w", encoding="utf-8") as f:
            json.dump(eval_data, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"ERROR: {e}")
        errors.append({"question": question, "error": str(e)})

print(f"\nCompleted: {len(eval_data)} successful, {len(errors)} errors")

Querying RAG system for 182 questions (starting from 0)...

[1/182] Kde nájdem Peňaženku zdravia? Je spoplatnená?... OK (4 contexts, 162 chars)
[2/182] Prečo je Peňaženka zdravia viazaná na mobilnú aplikáciu?... OK (4 contexts, 207 chars)
[3/182] Na čerpanie príspevkov z Peňaženky zdravia musím mať mobilnú aplikáciu,
alebo mi... OK (4 contexts, 80 chars)
[4/182] Ak som sa opäť vrátil do VšZP, môžem pre Peňaženku zdravia využívať svoje
staré ... OK (4 contexts, 107 chars)
[5/182] Som váš dlhoročný poistenec, prečo aj ja nemám nárok na Peňaženku
zdravia?... OK (4 contexts, 495 chars)
[6/182] Aktualizácia mobilnej aplikácie prebehne automaticky, alebo si novú verziu
musím... OK (4 contexts, 163 chars)
[7/182] Máte kontakt na podporu pre klientov? Mám problémy s inštaláciou mobilnej
apliká... OK (4 contexts, 161 chars)
[8/182] Prečo je potrebná aktivácia mobilnej aplikácie? Nie je to zbytočná strata
času?... OK (4 contexts, 117 chars)
[9/182] Ak mám nový telefón, musím si opätovne aktivova

## Save

In [8]:
eval_data_file = "eval_data_health_wallet.json"
with open(eval_data_file, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2)
print(f"Saved {len(eval_data)} entries to {eval_data_file}")

# Preview a sample entry
if eval_data:
    sample = eval_data[0]
    print("\n" + "=" * 60)
    print("Sample evaluation entry:")
    print("=" * 60)
    print(f"\nQuestion: {sample['user_input']}")
    print(f"\nRAG Response: {sample['response'][:300]}...")
    print(f"\nReference: {sample['reference'][:300]}...")
    print(f"\nRetrieved Contexts: {len(sample['retrieved_contexts'])} chunks")
    for j, ctx in enumerate(sample['retrieved_contexts'][:2]):
        print(f"  Context {j+1}: {ctx[:150]}...")

Saved 182 entries to eval_data_health_wallet.json

Sample evaluation entry:

Question: Kde nájdem Peňaženku zdravia? Je spoplatnená?

RAG Response: Peňaženku zdravia nájdete vo voľne dostupnej mobilnej aplikácii VšZP a v ePobočke. Využívať ju môžete po zaregistrovaní a aktivácii týchto elektronických služieb....

Reference: Peňaženku zdravia nájdete vo voľne dostupnej mobilnej aplikácii VšZP a v ePobočke. Využívať ju môžete po zaregistrovaní a aktivácii týchto elektronických služieb. Stačí si v mobilnej aplikácii alebo ePobočke zvoliť Peňaženku zdravia, v ktorej môžete podávať žiadosti o finančné príspevky, získať info...

Retrieved Contexts: 4 chunks
  Context 1: Peňaženka zdravia 
Ako funguje Peňaženka zdravia? 
Peňaženka zdravia je jedinečný benefitný produkt pre verných aj nových 
poistencov Všeobecnej zdrav...
  Context 2:   
1 z 17 
 
Všeobecné podmienky pre používanie produktu 
PEŇAŽENKA ZDRAVIA 
Úvod 
A. Tieto podmienky stanovujú kritériá pre používanie produktu Peňaž...
